In [553]:
import pandas as pd


action_hist = pd.read_csv('/Users/dmitrysologub/Desktop/АБ/Проект по АБ/action_hist.csv')

"""
Pre-experiment user activity data.

Action = firstOpen → first app launch  
Action = functionX → interaction with product features (X = feature id)
"""


payment_hist = pd.read_csv('/Users/dmitrysologub/Desktop/Проект по АБ/payments_hist.csv')

"""
Pre-experiment payment data.

paymentType = type1 → subscription purchase (historically only 400)  
paymentType = type2 → one-time purchase (always 200)
"""

action = pd.read_csv('/Users/dmitrysologub/Desktop/Проект по АБ/action.csv')

"""
Experimental user activity data.

Action = firstOpen → first app launch  
Action = functionX → interaction with product features
"""

payment = pd.read_csv('/Users/dmitrysologub/Desktop/Проект по АБ/payment.csv')

"""
Experimental payment data.

paymentType = type1 → subscription (160 or 400)  
paymentType = type2 → one-time purchase (always 200)
"""


experiments = pd.read_csv('/Users/dmitrysologub/Desktop/Проект по АБ/experiments.csv')

"""
Experiment assignment data.

Experiment = paymentChange → price experiment  
Experiment = actionChange → another experiment  
Group = Old → control  
Group = New → test  
"""

36058

In [878]:
# Group balance + contamination

# Group balance (Old / New)
experiments[experiments['Experiment'] == 'actionChange'].groupby('Group').size()


# Check if users appear in multiple groups
((experiments[experiments['Experiment'] == 'actionChange'])
 .groupby('uid')['Group']
 .nunique() > 1).sum()

0

In [880]:
# SRM check (sample ratio mismatch)

from scipy.stats import chisquare

# observed distribution
observed = [25000, 25001]

total_users = sum(observed)

# expected distribution (50/50)
expected = [total_users * 0.5, total_users * 0.5]

# chi-square test
stat, p_value = chisquare(f_obs=observed, f_exp=expected)

print(f"P-value: {p_value}")

# interpretation
if p_value < 0.01:
    print("SRM detected. The experiment may be biased.")
else:
    print("No SRM detected. Split is valid.")

P-value: 0.9964317993434099
SRM не обнаружен. Сплитование прошло успешно.


In [890]:
# Parallel experiments overlap

# users in both experiments
act_users = experiments[experiments['Experiment'] == 'actionChange']['uid']
pay_users = experiments[experiments['Experiment'] == 'paymentChange']['uid']

act_users = set(act_users)
pay_users = set(pay_users)

# intersection
print(len(act_users.intersection(pay_users)))

"""
The overlap between paymentChange and actionChange experiments is 3 users (~0.003%).

This impact is negligible, so paymentChange can be analyzed independently.
"""

50001


'\nThe overlap between paymentChange and actionChange experiments is 3 users (~0.003%).\n\nThis impact is negligible, so paymentChange can be analyzed independently.\n'

In [884]:
# Base dataset

# base users for paymentChange (1 row = 1 user)
df = experiments[experiments['Experiment'] == 'paymentChange'][['uid', 'Group']]

"""
Each user appears once → suitable for user-level metrics.
"""


# check aggregated payments
print(agg_pay.head())
print(agg_pay.columns)
print(len(agg_pay))

     uid  Payment
0  aaago      400
1  aaaii      360
2  aabfn      160
3  aabhx      200
4  aabka      200
Index(['uid', 'Payment'], dtype='object')
29530


In [888]:
# Merge payments

# basic checks
payment.info()
payment.nunique()

# total revenue per user
agg_pay = payment.groupby('uid')['Payment'].sum().reset_index()

# merge with users
result = pd.merge(df, agg_pay, on='uid', how='left')

# validation
print(len(result))     
print(result.columns)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36058 entries, 0 to 36057
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   uid          36058 non-null  object
 1   paymentType  36058 non-null  object
 2   Payment      36058 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 845.2+ KB
50001
Index(['uid', 'Group', 'Payment'], dtype='object')


In [860]:
# Metrics calculation

# subscription users (type1)
sub = payment[payment['paymentType'] == 'type1']['uid']

# subscription flag (1/0)
result['subscription'] = result['uid'].isin(sub).astype(int)

# replace NaN with 0
result['Payment'] = result['Payment'].fillna(0)

# conversion
conversion = result.groupby('Group')['subscription'].sum() / result.groupby('Group')['subscription'].count()

# ARPU
arpu = result.groupby('Group')['Payment'].sum() / result.groupby('Group')['Payment'].count()

In [830]:
# Conversion significance (z-test)

from statsmodels.stats.proportion import proportions_ztest

# total users
nobs = result.groupby('Group')['uid'].count()

# converters
count = result.groupby('Group')['subscription'].sum()

# fix order
count = count[['Old', 'New']]
nobs = nobs[['Old', 'New']]

# z-test
stat, pval = proportions_ztest(count, nobs)

print(f"P_value: {pval:.4f}")

# interpretation
alpha = 0.05

if pval < alpha:
    print("The difference in conversion is statistically significant.")
else:
    print("No statistically significant difference detected.")

P_value : 0.000000
Разница в конверсии статистически значима.


In [864]:
# ARPU significance (t-test)

from scipy import stats

# payment distributions
old_group = result[result['Group'] == 'Old']['Payment']
new_group = result[result['Group'] == 'New']['Payment']

# t-test
t_stat, p_val = stats.ttest_ind(old_group, new_group, equal_var=False)

print(f'T-statistic: {t_stat:.4f}')
print(f'P-value: {p_val:.4f}')

# interpretation
if p_val < 0.05:
    print("The difference in ARPU is statistically significant.")
else:
    print("The difference in ARPU may be due to random variation.")

T-статистика: -0.4213
P-value: 0.6735
Разница в ARPU может быть случайной
